# Biohub Drive Unzip and Verify

Bu notebook Drive'da duran Kaggle zip dosyasini temiz sekilde `data/` altina acar ve train/test sayimlarini dogrular. Varsayilan olarak `data/` klasorunu silmez; temizden baslamak istiyorsan flag'i bilincli olarak ac.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Ayarlar

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import time

PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
DATA_DIR = PROJECT_DIR / 'data'
COMPETITION_SLUG = 'biohub-cell-tracking-during-development'

# Zip'i acmak icin True kalsin.
RUN_EXTRACT = True

# Dikkat: True yaparsan DATA_DIR icindeki mevcut train/test/sample_submission silinir.
# Sen zaten siliyorum dedigin icin gerekirse True yapabilirsin, ama varsayilan guvenli.
CLEAN_DATA_DIR_BEFORE_EXTRACT = False

# Full veri icin beklenen train pair sayisi.
EXPECTED_TRAIN_PAIRS = 199

print('PROJECT_DIR:', PROJECT_DIR)
print('DATA_DIR:', DATA_DIR)

## 2. Zip Dosyasini Bul

In [ ]:
zip_candidates = []
search_dirs = [PROJECT_DIR, DATA_DIR, DATA_DIR / '_zip']
for d in search_dirs:
    if d.exists():
        zip_candidates.extend(sorted(d.glob('*.zip')))

# Daha derin arama: proje klasoru altinda zip baska yerdeyse yakala.
if not zip_candidates and PROJECT_DIR.exists():
    zip_candidates = sorted(PROJECT_DIR.rglob('*.zip'))

print('Zip candidates:')
for i, p in enumerate(zip_candidates):
    size_gb = p.stat().st_size / (1024 ** 3)
    print(f'{i}: {p}  ({size_gb:.2f} GiB)')

if not zip_candidates:
    raise FileNotFoundError('Zip bulunamadi. Zip dosyasini PROJECT_DIR, DATA_DIR veya DATA_DIR/_zip altina koy.')

# Birden fazla zip varsa en buyuk olani sec.
ZIP_PATH = max(zip_candidates, key=lambda p: p.stat().st_size)
print('\nSelected ZIP_PATH:', ZIP_PATH)

## 3. Temizleme ve Extract

In [ ]:
def remove_if_exists(path: Path):
    if path.is_dir():
        print('Removing directory:', path)
        shutil.rmtree(path)
    elif path.exists():
        print('Removing file:', path)
        path.unlink()


DATA_DIR.mkdir(parents=True, exist_ok=True)

if CLEAN_DATA_DIR_BEFORE_EXTRACT:
    remove_if_exists(DATA_DIR / 'train')
    remove_if_exists(DATA_DIR / 'test')
    remove_if_exists(DATA_DIR / 'sample_submission.csv')
    remove_if_exists(DATA_DIR / COMPETITION_SLUG)
else:
    print('CLEAN_DATA_DIR_BEFORE_EXTRACT=False, mevcut data icerigi silinmedi.')

if RUN_EXTRACT:
    start = time.time()
    print('Extracting zip... Bu Drive uzerinde uzun surebilir.')
    cmd = ['unzip', '-q', '-o', str(ZIP_PATH), '-d', str(DATA_DIR)]
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f'unzip failed with return code {result.returncode}')
    print(f'Extract complete in {(time.time() - start) / 60:.1f} min')
else:
    print('RUN_EXTRACT=False, extract atlandi.')

## 4. Dataset Base Dir Bul ve Gerekirse Normalize Et

In [ ]:
def find_dataset_base(data_dir: Path):
    candidates = [data_dir, data_dir / COMPETITION_SLUG]
    for c in candidates:
        if (c / 'train').exists() and (c / 'test').exists():
            return c
    raise FileNotFoundError(f'train/test bulunamadi: {candidates}')


BASE_DIR = find_dataset_base(DATA_DIR)
print('BASE_DIR:', BASE_DIR)

# Zip nested klasor actiysa kod yine calisir. Istersen sonra manuel normalize edebilirsin.
TRAIN_DIR = BASE_DIR / 'train'
TEST_DIR = BASE_DIR / 'test'
SAMPLE_SUBMISSION_PATH = BASE_DIR / 'sample_submission.csv'

print('TRAIN_DIR:', TRAIN_DIR)
print('TEST_DIR:', TEST_DIR)
print('sample_submission exists:', SAMPLE_SUBMISSION_PATH.exists())

## 5. Isim Eslesme Kontrolu

In [ ]:
train_zarr_paths = sorted(TRAIN_DIR.glob('*.zarr'))
train_geff_paths = sorted(TRAIN_DIR.glob('*.geff'))
test_zarr_paths = sorted(TEST_DIR.glob('*.zarr'))

zarr_ids = {p.stem for p in train_zarr_paths}
geff_ids = {p.stem for p in train_geff_paths}

print('train zarr:', len(zarr_ids))
print('train geff:', len(geff_ids))
print('train paired:', len(zarr_ids & geff_ids))
print('test zarr:', len(test_zarr_paths))
print('geff without zarr:', sorted(geff_ids - zarr_ids))
print('zarr without geff:', sorted(zarr_ids - geff_ids))

if len(zarr_ids & geff_ids) == EXPECTED_TRAIN_PAIRS and not (geff_ids - zarr_ids) and not (zarr_ids - geff_ids):
    print('PAIR CHECK OK')
else:
    print('PAIR CHECK NOT OK - eksik/fazla listelerine bak.')

## 6. Zarr Okunabilirlik Kontrolu

In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import biohub_baseline as bh

readable = []
errors = []
for p in train_zarr_paths:
    try:
        img = bh.open_image_zarr(p, print_info=False)
        # Metadata + tiny read. Full volume yuklenmez.
        _ = img[0, 0, 0:4, 0:4]
        readable.append(p.stem)
    except Exception as e:
        errors.append({'sample_id': p.stem, 'error': repr(e)})

print('readable train zarr:', len(readable))
print('zarr read errors:', len(errors))
for row in errors[:50]:
    print(row['sample_id'], row['error'])

if len(readable) == EXPECTED_TRAIN_PAIRS and not errors:
    print('ZARR READ CHECK OK')
else:
    print('ZARR READ CHECK NOT OK')

## 7. Final Beklenen Cikti

Temiz full dataset icin beklenen:

```text
train zarr: 199
train geff: 199
train paired: 199
test zarr: 4
geff without zarr: []
zarr without geff: []
readable train zarr: 199
zarr read errors: 0
```